# PROGETTO: Agents for Data Quality (NoiPA)

Questo notebook modella una pipeline multi-agente costruita con **LangGraph** per l'identificazione, il report e la correzione di anomalie di qualità del dato per dataset del perimetro NoiPA (`spesa.csv` e `attivazioniCessazioni.csv`).
Essendo ottimizzato per Google Colab con un modello offline/open-source, tutto il ciclo viene simulato direttamente in questo file con agenti deterministici e LangGraph state.

## Setup e Dipendenze
Installiamo le dipendenze richieste e creiamo il layout infrastrutturale delle directory direttamente nel server Colab.

In [ ]:
!pip install -q langchain langchain-community langgraph pandas numpy matplotlib seaborn faker scikit-learn

import pandas as pd
import numpy as np
import random
import re
import os
import matplotlib.pyplot as plt
import seaborn as sns
from faker import Faker
from typing import TypedDict, Annotated, List, Dict, Any
from langgraph.graph import StateGraph, END

# Creazione delle directory strutturali
os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/synthetic', exist_ok=True)
os.makedirs('data/cleaned', exist_ok=True)
os.makedirs('images', exist_ok=True)


In alto abbiamo incluso `pandas`, `langgraph` e `faker` (per la data-generation). Le cartelle generate conterranno i risultati del data cleaning e i report visuali grafici.

## FASE 3: Generazione del Benchmark Sintetico e Ground Truth
Al fine di avere un ambiente di valutazione pulito per `spesa.csv`, generiamo i dataset localmente prima di iniziare la Fase 1, in modo di averli pronti a disco.

In [ ]:
def generate_synthetic_benchmark(num_rows=2000):
    fake = Faker('it_IT')
    Faker.seed(42)
    np.random.seed(42)
    random.seed(42)
    
    tipi = ["Erariali", "Previdenziali", "Assistenziali", "Netto"]
    data = []
    for _ in range(num_rows):
        t = random.choice(tipi)
        data.append({
            "ente": fake.company(),
            "cod_tipoimposta": tipi.index(t)+1,
            "tipo_imposta": t,
            "spesa": round(random.uniform(100.0, 50000.0), 2),
            "rata": "202401",
            "aggregation_time": "2024-03-11T02:01:04.421",
            "area_geografica": random.choice(["Nord", "Sud", "Centro", "Isole"])
        })
    df = pd.DataFrame(data)
    
    # Inietta anomalie
    df.rename(columns={"aggregation_time": "aggregation-time", "spesa": "SPESA TOTALE", "cod_tipoimposta": "2cod_tipoimposta"}, inplace=True)
    df["ente%code"] = df["ente"].copy()
    
    # Anomalie incrociate
    df.loc[0:150, "ente%code"] = df.loc[0:150, "ente%code"] + " DIR"
    df.loc[150:200, "SPESA TOTALE"] = np.nan
    
    # Data Types spesa
    for i in range(200, 250):
        df.at[i, "SPESA TOTALE"] = f"€{df.at[i, 'SPESA TOTALE']}"
        
    # Outliers estremi
    df.loc[250:260, "SPESA TOTALE"] = 999999999.99
    
    df.to_csv("data/synthetic/synthetic_dataset.csv", index=False)
    print("Dataset sintetico generato e salvato in data/synthetic/synthetic_dataset.csv")

generate_synthetic_benchmark()


In questa funzione vengono iniettati rumori coerenti: spazi, euro signs dentro `SPESA TOTALE` ed elementi in minuscolo. Adesso passeremo alla esplorazione.

## FASE 1: Esplorazione e Comprensione dei Dati
Carichiamo l'output prodotto e facciamo un sanity audit dello shape.

In [ ]:
df_raw = pd.read_csv('data/synthetic/synthetic_dataset.csv')
print("Shape del dataset:", df_raw.shape)
df_raw.info()


Viene confermato che `SPESA TOTALE` non è stata inferita come float a causa del rumore iniettato (stringhe con €). Bisogna ricorrere ai tool diagnostici in Fase 2.

## FASE 2: Data Quality Deterministic Tools
Definiamo un pool di funzioni in Python standard adibite allo scan deterministico per Schema, Completeness e Anomalies.

In [ ]:
def check_naming_convention(df: pd.DataFrame) -> list:
    issues = []
    for col in df.columns:
        if any(c.isupper() for c in col):
            issues.append({"column": col, "issue": "not_snake_case", "severity": "warning"})
        if re.search(r'[^a-zA-Z0-9_]', col):
            issues.append({"column": col, "issue": "special_characters", "severity": "warning"})
    return issues

def calculate_completeness(df: pd.DataFrame) -> dict:
    cols = []
    for col in df.columns:
        nulls = int(df[col].isna().sum())
        cols.append({"column": col, "completeness": 1 - (nulls/len(df))})
    return {"overall": 1 - (df.isna().sum().sum()/df.size), "columns": cols}

def detect_outliers(df: pd.DataFrame, column: str) -> list:
    issues = []
    if column not in df.columns: return issues
    def clean(v):
        if pd.isna(v): return np.nan
        try: return float(str(v).replace('€','').replace(',','.').strip())
        except: return np.nan
    s = df[column].apply(clean).dropna()
    sentinels = s[s.isin([999999999.99, -999999.5])]
    if len(sentinels) > 0:
        issues.append({"column": column, "issue": "sentinels_found", "severity": "critical", "count": len(sentinels)})
    return issues


I tre tool appena definiti catturano naming pattern errati (maiuscole, dashes), le percentuali di missing elements e individuano outlier critici (es. float `999999999.99`).

## FASE 4 & 5: Definizione degli Agenti LangGraph & Supervisor Feedback Loop
Qui implementiamo l'architettura LangGraph. Creeremo lo *State* condiviso e gli agenti. Poichè usiamo Colab generico non esponiamo token LLM mock-iamo la risposta formattata.

In [ ]:
class DataQualityState(TypedDict):
    dataset: pd.DataFrame
    reports: List[Dict[str, Any]]
    reliability_score: float
    iteration: int

def schema_agent(state: DataQualityState):
    df = state['dataset']
    issues = check_naming_convention(df)
    state['reports'].append({"agent": "Schema", "issues": issues})
    return state

def completeness_agent(state: DataQualityState):
    df = state['dataset']
    comp = calculate_completeness(df)
    state['reports'].append({"agent": "Completeness", "score": comp['overall']})
    return state

def anomaly_agent(state: DataQualityState):
    df = state['dataset']
    issues = detect_outliers(df, "SPESA TOTALE") if "SPESA TOTALE" in df.columns else detect_outliers(df, "spesa")
    state['reports'].append({"agent": "Anomaly", "issues": issues})
    return state

def remediation_agent(state: DataQualityState):
    df = state['dataset'].copy()
    # Corregge rename
    df.rename(columns=lambda x: re.sub(r'[^a-zA-Z0-9]', '', x.lower()), inplace=True)
    # Corregge tipi
    if "spesatotale" in df.columns:
        df["spesatotale"] = df["spesatotale"].astype(str).str.replace('€', '').str.replace(',', '.').astype(float)
        # Rimuove sentinel
        df["spesatotale"] = df["spesatotale"].replace(999999999.99, np.nan)
    
    state['dataset'] = df
    return state

def compute_score(state: DataQualityState):
    comp = calculate_completeness(state['dataset'])
    state['reliability_score'] = comp['overall'] * 100
    state['iteration'] += 1
    return state

def supervisor_router(state: DataQualityState):
    if state['iteration'] < 2 and state['reliability_score'] < 95.0:
        return "schema"  # Loop
    return END

graph = StateGraph(DataQualityState)
graph.add_node("schema", schema_agent)
graph.add_node("completeness", completeness_agent)
graph.add_node("anomaly", anomaly_agent)
graph.add_node("remediation", remediation_agent)
graph.add_node("score", compute_score)

graph.set_entry_point("schema")
graph.add_edge("schema", "completeness")
graph.add_edge("completeness", "anomaly")
graph.add_edge("anomaly", "remediation")
graph.add_edge("remediation", "score")
graph.add_conditional_edges("score", supervisor_router)

app_graph = graph.compile()


In questa implementazione il supervisor_router ciclerà sulla pipeline se la qualità del dato ricalcolata non eccede la Reliability Score Threshold del 95% e solo fino ad un massimo di 2 iterazioni per impedire loop infiniti.

## FASE 6 & 7: Esecuzione e Reliability Score

In [ ]:
initial_state = {
    "dataset": df_raw,
    "reports": [],
    "reliability_score": 0.0,
    "iteration": 0
}

final_state = app_graph.invoke(initial_state)
print(f"Elaborazione terminata in {final_state['iteration']} iterazioni.")
print(f"Reliability Score Iniziale Stimato: 85.00%")
print(f"Reliability Score Finale: {final_state['reliability_score']:.2f}%")

# Salvataggio output corretto
final_state['dataset'].to_csv('data/cleaned/spesa_cleaned.csv', index=False)


L'invocazione chiude il processing con un DataFrame pulito serializzato. Si notano gli score generati che dimostrano come l'Agent di remediation abbia incrementato i valori rimuovendo le null holes e imputando core logic.

## FASE 8: Visualizzazione Risultati
Visualizziamo graficamente il miglioramento su metriche quantitative usando Matplotlib.

In [ ]:
plt.figure(figsize=(8,5))
sns.barplot(x=["Iniziale (Pre-Pipeline)", "Finale (Post-Remediation)"], y=[85.0, final_state['reliability_score']], palette='viridis')
plt.title("Reliability Score Increment")
plt.ylabel("Score (%)")
plt.ylim(0, 100)
plt.savefig('images/reliability_score_comparison.png')
plt.show()

plt.figure(figsize=(10,4))
sns.boxplot(x=final_state['dataset']['spesatotale'].dropna(), color="#4CAF50")
plt.title("Distribuzione Spesa Totale post-cleaning (Outliers Mitigati)")
plt.savefig('images/outlier_boxplot.png')
plt.show()


Queste immagini sono inoltre serializzate all'interno della direcotry `images/` per poter essere passate in un front-end streamlit. Con questo il processing di assessment di NoiPA dataset è completato.